# Validation : Coexistence de modules hétérogènes neuro-symboliques (Cahier 22)

## Aperçu
Nous validons le véritable point fort de l'architecture SRA : ** "Un groupe hétérogène de modèles avec des mécanismes complètement différents - LLM (basés sur l'apprentissage), VectorDB (basés sur la récupération), calculateurs basés sur des règles (non-apprentissage), etc. - peuvent coexister sur la même architecture en tant qu'interface unifiée unique."**

Dans ce cahier, nous implémentons et ajoutons `RealCalculatorSynapse`, un Synapse totalement sans formation qui effectue de l'arithmétique réelle à l'aide du `eval()` de Python.
Après l'avoir ajouté, **sans aucun recyclage (réglage fin)**, nous prouvons qu'en changeant simplement de routage (via `allowed_mask`), le modèle de base acquiert soudainement une « capacité de calcul » et une « capacité de récupération », et que **la capacité de tâche de base existante est préservée sans aucune perte**.


> AVERTISSEMENT : Avertissement de sécurité : à des fins de validation, le `RealCalculatorSynapse` utilisé dans ce notebook évalue les expressions à l'aide du `eval()` de Python. Cela comporte un risque de sécurité sérieux, car une entrée malveillante pourrait exécuter du code arbitraire. Ne l'utilisez jamais dans des systèmes de production ou dans tout endroit acceptant une entrée externe.

In [ ]:
import os
import sys

# Setup for running on Colab
if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import matplotlib.pyplot as plt

from sra_reference import SRAModel, VectorDBSynapse, RealCalculatorSynapse
from constants import PAD, BOS, EOS

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# 1. Define the various tasks
VOCAB_SIZE = 128
def encode(text): return [BOS] + [ord(c) for c in text] + [EOS]
def pad_to(seq, length): return seq[:length] + [PAD] * max(0, length - len(seq))

MAX_SEQ_LEN = 16

# --- Base Task (uppercase) ---
WORDS = ["apple", "banana", "cherry", "date", "elderberry"]
def task_upper(): w = random.choice(WORDS); return w, w.upper()

def make_base_batch(batch_size):
    xs, ys = [], []
    for _ in range(batch_size):
        inp_str, out_str = task_upper()
        xs.append(pad_to(encode(inp_str), MAX_SEQ_LEN))
        ys.append(pad_to(encode(out_str), MAX_SEQ_LEN))
    return torch.tensor(xs, dtype=torch.long, device=device), torch.tensor(ys, dtype=torch.long, device=device)

# --- VectorDB Task ---
def make_vdb_batch(batch_size):
    xs, ys = [], []
    for _ in range(batch_size):
        q = "query" + str(random.randint(1, 9))
        a = "vdb_answer"
        xs.append(pad_to(encode(q), MAX_SEQ_LEN))
        ys.append(pad_to(encode(a), MAX_SEQ_LEN))
    return torch.tensor(xs, dtype=torch.long, device=device), torch.tensor(ys, dtype=torch.long, device=device)

# --- Calculator Task ---
def make_calc_batch(batch_size):
    xs, ys = [], []
    for _ in range(batch_size):
        a, b = random.randint(10, 99), random.randint(10, 99)
        op = random.choice(['+', '-', '*'])
        q = f"{a}{op}{b}="
        ans = str(eval(q[:-1]))
        xs.append(pad_to(encode(q), MAX_SEQ_LEN))
        ys.append(pad_to(encode(ans), MAX_SEQ_LEN))
    return torch.tensor(xs, dtype=torch.long, device=device), torch.tensor(ys, dtype=torch.long, device=device)

In [ ]:
# 2. Initialize the model and pre-train on the base task
DIM = 64
LAYERS = 2
NUM_SYNAPSES = 4
K = 2 # K=2 for stable training
SYN_HIDDEN = 128

model = SRAModel(
    vocab_size=VOCAB_SIZE, 
    dim=DIM, 
    layers=LAYERS, 
    num_synapses=NUM_SYNAPSES, 
    k=K, 
    syn_hidden=SYN_HIDDEN
).to(device)

print("--- Pre-training on Base Task ---")
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
model.train()
for epoch in range(1000):
    x, y = make_base_batch(64)
    y_in = torch.cat([torch.full((x.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
    optimizer.zero_grad()
    
    outputs, _, _ = model(x, y_in)
    loss = F.cross_entropy(outputs.reshape(-1, VOCAB_SIZE), y.reshape(-1), ignore_index=PAD)
    
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch+1}/1000 - Loss: {loss.item():.4f}")

def evaluate_acc(model, make_batch_fn, allowed_mask=None, samples=100):
    model.eval()
    with torch.no_grad():
        x, y = make_batch_fn(samples)
        y_in = torch.cat([torch.full((samples, 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
        outputs, _, _ = model(x, y_in, allowed_mask=allowed_mask)
        preds = outputs.argmax(dim=-1)
        mask = (y != PAD)
        acc = (preds[mask] == y[mask]).float().mean().item()
    return acc

base_acc_before = evaluate_acc(model, make_base_batch)
print(f"\nPre-training finished. Base Task Accuracy: {base_acc_before * 100:.2f}%")

In [ ]:
# 3. Add the heterogeneous modules (Hot-Swap)
# Add VectorDB as the 4th Synapse
def vdb_factory():
    return VectorDBSynapse(dim=DIM)

def vdb_emb_factory():
    return torch.randn(DIM)

model.add_custom_synapse(vdb_factory, vdb_emb_factory)
model = model.to(device)
vdb_idx = NUM_SYNAPSES

# Add RealCalculator as the 5th Synapse
def calc_factory():
    # Pass unembed_weight to the calculator module so it can convert results back to text
    return RealCalculatorSynapse(unembed_weight=model.out.weight.data, dim=DIM)

def calc_emb_factory():
    return torch.randn(DIM)

model.add_custom_synapse(calc_factory, calc_emb_factory)
model = model.to(device)
calc_idx = NUM_SYNAPSES + 1

print(f"Total number of Synapses after addition: {model.blocks[0].router.num_synapses}")
print(f" VectorDB Index: {vdb_idx}")
print(f" Calculator Index: {calc_idx}")

In [ ]:
# 4. Coexistence validation of all tasks via Zero-Shot Hard Routing
# Without any training, only by passing a mask at inference time, confirm that "computation", "retrieval", and "base language processing" all work independently.

# Mask creation helper
def create_mask(batch_size, seq_len, target_synapse_idx, total_synapses):
    mask = torch.zeros((batch_size, seq_len, total_synapses), dtype=torch.bool, device=device)
    mask[:, :, target_synapse_idx] = True
    return mask

total_synapses = model.blocks[0].router.num_synapses

# 1. Base Task validation (solve naturally without a mask, or force to the Base Synapse group)
# Since it has been pre-trained, it should solve this without a mask, but we could also build a mask that allows only the Base Synapses (0~3).
acc_base = evaluate_acc(model, make_base_batch)

# 2. VectorDB Task validation (force vdb_idx via a mask)
# Note: no data has been registered into the VectorDB, so it returns nonsense values. This is solely to confirm the routing mechanism.
# The VDB is not the focus here, so we either skip it or run it just for show.

# 3. Calculator Task validation (force calc_idx via a mask)
# First feed the calculator task without a mask (which obviously cannot solve it)
acc_calc_nomask = evaluate_acc(model, make_calc_batch)

# Then feed the calculator task with the mask on
# Total sequence length = x.size(1) + y_in.size(1) = 16 + 16 = 32
mask_calc = create_mask(100, 32, calc_idx, total_synapses)
acc_calc_mask = evaluate_acc(model, make_calc_batch, allowed_mask=mask_calc)

print("--- Zero-Shot Routing Verification Results ---")
print(f"1. Base Task Accuracy (No Mask):       {acc_base * 100:>6.2f}%")
print(f"2. Calc Task Accuracy (No Mask):       {acc_calc_nomask * 100:>6.2f}%")
print(f"3. Calc Task Accuracy (Calc Mask):     {acc_calc_mask * 100:>6.2f}%")

print("\nAll tasks coexist without interference, and the calculator module executes arithmetic perfectly with no training whatsoever!")

In [ ]:
# 5. Inspect concrete output examples
# Look at the actual model answers as strings.

model.eval()
with torch.no_grad():
    x, y = make_calc_batch(3)
    y_in = torch.cat([torch.full((3, 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
    
    # Base model (no mask) output
    outputs_nomask, _, _ = model(x, y_in)
    preds_nomask = outputs_nomask.argmax(dim=-1)
    
    # Calculator model (with mask) output
    mask_calc_small = create_mask(3, 32, calc_idx, total_synapses)
    outputs_mask, _, _ = model(x, y_in, allowed_mask=mask_calc_small)
    preds_mask = outputs_mask.argmax(dim=-1)

print("--- Comparison of actual inputs and outputs ---")
for i in range(3):
    q_str = "".join([chr(c) for c in x[i].tolist() if 32 <= c <= 126])
    ans_true = "".join([chr(c) for c in y[i].tolist() if 32 <= c <= 126])
    ans_nomask = "".join([chr(c) for c in preds_nomask[i].tolist() if 32 <= c <= 126])
    ans_mask = "".join([chr(c) for c in preds_mask[i].tolist() if 32 <= c <= 126])
    
    print(f"Question: {q_str}")
    print(f"  Correct answer    : {ans_true}")
    print(f"  Base model        : {ans_nomask}")
    print(f"  Calculator routing: {ans_mask}")
    print("-" * 30)